# RelBench Relational Benchmark with GNNs

Relational Prediction on RelBench: End-to-end relational table learning using multi-relational graph convolutions. This notebook implements the approach with `HeteroGNN` inside a `K3RelationalGNN` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `HeteroGNN` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "RelBench Relational Benchmark with Heterogeneous GNN"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Relational GNN Model
class K3RelationalGNN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = k3_layers.SAGEConv(in_channels, hidden_channels)
        self.conv2 = k3_layers.SAGEConv(hidden_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

k3_model = K3RelationalGNN(in_channels=16, hidden_channels=32, out_channels=1)

# 2. Forward pass test
num_nodes = 30
dummy_x = keras.random.normal((num_nodes, 16))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")

out = k3_model(dummy_x, dummy_edges)
print(f"RelBench prediction output shape: {out.shape}")

print("\n✓ K3-Node RelBench execution completed successfully!")